In [221]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from collections import Counter

In [222]:
reviews = pd.read_csv(
    "../datasets/processed/silver_order_reviews_en.csv"
)

In [223]:
reviews.shape

(99224, 10)

In [224]:
reviews.columns.tolist()

['id_order_reviews',
 'review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_title_en',
 'review_comment_message',
 'review_comment_message_en',
 'review_creation_date',
 'review_answer_timestamp']

In [225]:
reviews.head()

,id_order_reviews,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
0,1,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,2,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,3,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,4,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,NaN,recebi bem antes do prazo estipulado.,i received it well before the promised deadline.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,5,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,NaN,parabns lojas lannister adorei comprar pela in...,"congratulations, lannister stores! i loved buy...",2018-03-01 00:00:00,2018-03-02 10:26:53


In [226]:
reviews[[
    "review_score",
    "review_comment_message",
    "review_comment_message_en"
]].isna().sum()

review_score                     0
review_comment_message       58338
review_comment_message_en    58338
dtype: int64

In [227]:
comment_text = reviews["review_comment_message_en"].fillna("").astype(str)

empty_comments = (comment_text.str.strip() == "").sum()

empty_comments

np.int64(58338)

In [228]:
comments = reviews.loc[
    comment_text.str.strip() != "",
    [
        "id_order_reviews",
        "review_id",
        "order_id",
        "review_score",
        "review_comment_message_en"
    ]
].copy()

In [229]:
comments[
    [
        "review_score",
        "review_comment_message_en"
    ]
].head(20)

,review_score,review_comment_message_en
3,5,i received it well before the promised deadline.
4,5,"congratulations, lannister stores! i loved buy..."
9,4,"efficient device. on the site, the devices bra..."
12,4,"but a little bit, its lagging... for the price..."
15,5,"reliable seller, product ok, and delivery befo..."
16,2,"i would like to know what happened, i always r..."
19,1,terrible
22,5,store
24,5,thank you for the attention you gave me.
27,5,the purchase was made easily. the delivery was...


In [230]:
comments["word_count"] = (
    comments["review_comment_message_en"]
    .str.split()
    .str.len()
)

comments["char_count"] = (
    comments["review_comment_message_en"]
    .str.len()
)

comments[["word_count", "char_count"]].describe()

,word_count,char_count
count,40886.000000,40886.000000
mean,13.156851,74.480189
std,10.691490,57.306210
min,1.000000,1.000000
25%,5.000000,31.000000
50%,10.000000,59.000000
75%,18.000000,103.000000
max,56.000000,282.000000


In [231]:
comments.groupby("review_score").size().reset_index(
    name="comment_count"
)

,review_score,comment_count
0,1,8740
1,2,2143
2,3,3555
3,4,5956
4,5,20492


In [232]:
comments.groupby("review_score")["word_count"].agg(
    ["count", "mean", "median"]
).reset_index()

,review_score,count,mean,median
0,1,8740,19.858009,18.0
1,2,2143,19.304246,17.0
2,3,3555,16.512799,14.0
3,4,5956,11.877267,9.0
4,5,20492,9.445589,7.0


In [233]:
comments.loc[
    comments["review_score"] == 1,
    ["review_score", "review_comment_message_en"]
].sample(15, random_state=42)

,review_score,review_comment_message_en
72196,1,the product has not yet reached my hands i ask...
83137,1,it was the first time i made a purchase online...
91980,1,i bought identical products with individual sh...
18684,1,i ordered single elegance mosquito nets and qu...
36173,1,as of today i have not received my product and...
46333,1,it is the fourth lannister key knife i have bo...
38850,1,i have not received my product yet. it says it...
16820,1,"the product arrived missing two pieces, and i ..."
25635,1,i have not received my product yet!!! there is...
81285,1,there should be a certain level of care with d...


In [234]:
comments.loc[
    comments["review_score"] == 5,
    ["review_score", "review_comment_message_en"]
].sample(15, random_state=42)

,review_score,review_comment_message_en
52025,5,"good quality product, delivered on the agreed ..."
72658,5,the product was on the site with a deadline of...
92644,5,i simply loved my product!
11706,5,the order arrived right on time.
4614,5,i liked it a lot they delivered earlier than e...
84816,5,product delivered very quickly.
64406,5,"i am delighted with my cans, they are much pre..."
58944,5,beautiful cushion!!
73500,5,i really enjoyed making this purchase with you...
87624,5,"i recommend the store, as it fulfilled what wa..."


In [299]:
def clean_text(text):
    text = text.lower()

    # Normalize common contractions
    text = text.replace("didn't", "didnt")
    text = text.replace("doesn't", "doesnt")
    text = text.replace("don't", "dont")
    text = text.replace("can't", "cant")
    text = text.replace("couldn't", "couldnt")
    text = text.replace("wouldn't", "wouldnt")
    text = text.replace("haven't", "havent")
    text = text.replace("hasn't", "hasnt")
    text = text.replace("isn't", "isnt")
    text = text.replace("wasn't", "wasnt")
    text = text.replace("weren't", "werent")
    text = text.replace("I'm", "im")
    text = text.replace("I've", "ive")
    text = text.replace("I'll", "ill")

    # Remove remaining punctuation
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [298]:
comments["clean_text"] = (
    comments["review_comment_message_en"]
    .apply(clean_text)
)

In [237]:
test_comments = [
    "I didn't receive my product",
    "The product doesn't work",
    "I haven't received my order",
    "The seller wasn't helpful",
    "I don't recommend this product"
]

for text in test_comments:
    cleaned = clean_text(text)

    print(f"\nOriginal: {text}")
    print(f"Cleaned:  {cleaned}")
    print(f"Topics:   {classify_topics(cleaned, topic_keywords)}")


Original: I didn't receive my product
Cleaned:  i didnt receive my product
Topics:   ['Delivery & Logistics']

Original: The product doesn't work
Cleaned:  the product doesnt work
Topics:   ['Product & Quality']

Original: I haven't received my order
Cleaned:  i havent received my order
Topics:   ['Delivery & Logistics', 'Order & Purchase']

Original: The seller wasn't helpful
Cleaned:  the seller wasnt helpful
Topics:   ['Seller & Service']

Original: I don't recommend this product
Cleaned:  i dont recommend this product
Topics:   []


In [238]:
word_counter = Counter(
    word
    for text in comments["clean_text"]
    for word in text.split()
)

word_counter.most_common(30)

[('the', 45073),
 ('i', 30618),
 ('it', 19362),
 ('product', 18377),
 ('and', 15990),
 ('to', 9123),
 ('a', 8734),
 ('was', 8407),
 ('is', 7835),
 ('very', 7254),
 ('not', 6846),
 ('arrived', 6576),
 ('deadline', 6422),
 ('delivery', 6244),
 ('good', 6237),
 ('of', 5177),
 ('delivered', 5126),
 ('before', 5125),
 ('received', 5071),
 ('but', 4822),
 ('recommend', 4745),
 ('in', 4686),
 ('on', 4526),
 ('for', 4521),
 ('with', 4200),
 ('my', 4194),
 ('as', 3442),
 ('have', 3389),
 ('great', 3188),
 ('time', 3082)]

In [239]:
stop_words = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but",
    "by", "for", "from", "had", "has", "have", "he", "her",
    "here", "him", "his", "i", "if", "in", "is", "it", "its",
    "me", "my", "of", "on", "or", "our", "ours", "she", "so",
    "that", "the", "their", "them", "there", "they", "this",
    "to", "too", "was", "we", "were", "what", "when", "which",
    "who", "will", "with", "you", "your", "yours"
}

filtered_words = [
    word
    for text in comments["clean_text"]
    for word in text.split()
    if word not in stop_words
]

word_counter_filtered = Counter(filtered_words)

word_counter_filtered.most_common(30)

[('product', 18377),
 ('very', 7254),
 ('not', 6846),
 ('arrived', 6576),
 ('deadline', 6422),
 ('delivery', 6244),
 ('good', 6237),
 ('delivered', 5126),
 ('before', 5125),
 ('received', 5071),
 ('recommend', 4745),
 ('great', 3188),
 ('time', 3082),
 ('bought', 3054),
 ('one', 3023),
 ('only', 2731),
 ('quality', 2718),
 ('excellent', 2686),
 ('well', 2572),
 ('fast', 2448),
 ('store', 2416),
 ('everything', 2154),
 ('am', 2013),
 ('came', 1997),
 ('purchase', 1979),
 ('did', 1923),
 ('expected', 1711),
 ('loved', 1707),
 ('liked', 1588),
 ('yet', 1580)]

In [240]:
bigram_counter = Counter()

for text in comments["clean_text"]:
    words = [
        word
        for word in text.split()
        if word not in stop_words
    ]
    
    bigrams = zip(words, words[1:])
    
    bigram_counter.update(
        " ".join(bigram)
        for bigram in bigrams
    )

bigram_counter.most_common(30)

[('before deadline', 3265),
 ('very good', 2924),
 ('product arrived', 2068),
 ('did not', 1838),
 ('product delivered', 1578),
 ('fast delivery', 1416),
 ('received product', 1338),
 ('arrived before', 1282),
 ('well before', 1136),
 ('delivered before', 992),
 ('good product', 932),
 ('great product', 917),
 ('within deadline', 911),
 ('not receive', 857),
 ('product not', 771),
 ('post office', 767),
 ('well packaged', 761),
 ('not received', 735),
 ('excellent quality', 723),
 ('highly recommend', 684),
 ('not delivered', 675),
 ('very satisfied', 654),
 ('only received', 649),
 ('arrived well', 603),
 ('receive product', 594),
 ('perfect condition', 578),
 ('delivered time', 577),
 ('super fast', 570),
 ('quality product', 569),
 ('am waiting', 546)]

In [241]:
def get_bigrams_by_score(df, score):
    counter = Counter()

    texts = df.loc[
        df["review_score"] == score,
        "clean_text"
    ]

    for text in texts:
        words = [
            word
            for word in text.split()
            if word not in stop_words
        ]

        bigrams = zip(words, words[1:])

        counter.update(
            " ".join(bigram)
            for bigram in bigrams
        )

    return counter

In [242]:
bigrams_score_1 = get_bigrams_by_score(comments, 1)

bigrams_score_1.most_common(30)

[('did not', 1283),
 ('not receive', 694),
 ('received product', 635),
 ('not received', 580),
 ('product not', 572),
 ('not delivered', 516),
 ('receive product', 484),
 ('havent received', 397),
 ('only received', 379),
 ('do not', 362),
 ('am waiting', 331),
 ('post office', 310),
 ('does not', 277),
 ('only one', 273),
 ('not recommend', 265),
 ('product arrived', 250),
 ('bought two', 238),
 ('product yet', 219),
 ('delivery deadline', 211),
 ('still havent', 203),
 ('would like', 198),
 ('received one', 172),
 ('money back', 170),
 ('product did', 168),
 ('not arrive', 167),
 ('bought product', 163),
 ('product delivered', 154),
 ('no one', 150),
 ('more than', 148),
 ('still waiting', 148)]

In [243]:
bigrams_score_5 = get_bigrams_by_score(comments, 5)

bigrams_score_5.most_common(30)

[('before deadline', 2670),
 ('very good', 2259),
 ('product arrived', 1234),
 ('fast delivery', 1197),
 ('product delivered', 1029),
 ('arrived before', 1020),
 ('well before', 976),
 ('delivered before', 808),
 ('great product', 781),
 ('highly recommend', 634),
 ('excellent quality', 631),
 ('well packaged', 628),
 ('very satisfied', 595),
 ('good product', 581),
 ('within deadline', 552),
 ('super fast', 501),
 ('arrived well', 501),
 ('excellent product', 469),
 ('perfect condition', 458),
 ('product excellent', 427),
 ('very fast', 419),
 ('quality product', 418),
 ('before expected', 412),
 ('delivered time', 370),
 ('than expected', 365),
 ('good quality', 348),
 ('earlier than', 345),
 ('very well', 341),
 ('received product', 337),
 ('product very', 315)]

In [244]:
topic_keywords = {
    "Delivery & Logistics": [
        "delivery",
        "delivered",
        "deliver",
        "arrived",
        "arrive",
        "arrival",
        "deadline",
        "shipping",
        "shipment",
        "received",
        "receive",
        "waiting",
        "wait",
        "post office",
        "late",
        "delay",
        "delayed",
        "quickly",
        "fast",
        "on time",
        "before deadline",
        "not received",
        "not delivered",
        "not arrive",
        "did not arrive",
        "didnt arrive",
        "still waiting",
        "when it will arrive"
    ],

   "Product & Quality": [
    "quality",
    "condition",
    "size",
    "material",
    "damaged product",
    "product damaged",
    "product was damaged",
    "item damaged",
    "arrived damaged",
    "damage to the product",
    "broken product",
    "product is broken",
    "product was broken",
    "defective product",
    "product is defective",
    "defect in the product",
    "defect",
    "thin",
    "thick",
    "thicker",
    "leather",
    "fringes",
    "fringe",
    "screen",
    "cable",
    "battery",
    "batteries",
    "holder",
    "holders",
    "pieces",
    "piece",
    "holes",
    "hole",
    "aluminum",
    "inferior",
    "burst",
    "tear",
    "torn",
    "broke",
    "doesnt work",
    "does not work",
    "not work",
    "perfect condition",
    "product quality",
    "good quality",
    "excellent quality",
    "poor quality",
    "high quality",
    "good product",
    "great product",
    "beautiful product",
    "wonderful product",
    "excellent product",
    "product is very good"
],

    "Seller & Service": [
        "seller",
        "store",
        "service",
        "support",
        "customer service",
        "responsive",
        "response",
        "attention",
        "helpful",
        "answer",
        "answered"
    ],

    "Order & Purchase": [
        "order",
        "purchase",
        "bought",
        "buy",
        "purchased",
        "shopping",
        "paid",
        "payment",
        "came wrong",
        "wrong item",
        "missing item",
        "missing items",
        "only came",
        "didnt come",
        "did not come"
    ],

    "Packaging": [
    "packaging",
    "packaged",
    "well packaged",
    "very well packaged",
    "adequately packaged",
    "properly packaged",
    "poorly packaged",
    "badly packaged",
    "damaged packaging",
    "packaging damaged",
    "packaging was",
    "packaging is",
    "packaging condition"
]
}

In [245]:
boundary_test_comments = [
    "the packaging was damaged",
    "the product was damaged",
    "the product arrived damaged",
    "the packaging was very good",
    "the product is broken",
    "the box was damaged"
]

for text in boundary_test_comments:
    cleaned = clean_text(text)

    print(f"\nComment: {text}")
    print(f"Topics: {classify_topics(cleaned, topic_keywords)}")


Comment: the packaging was damaged
Topics: ['Packaging']

Comment: the product was damaged
Topics: ['Product & Quality']

Comment: the product arrived damaged
Topics: ['Delivery & Logistics', 'Product & Quality']

Comment: the packaging was very good
Topics: ['Packaging']

Comment: the product is broken
Topics: ['Product & Quality']

Comment: the box was damaged
Topics: []


In [246]:
def classify_topics(text, topic_keywords):
    topics = []

    for topic, keywords in topic_keywords.items():
        for keyword in keywords:
            pattern = rf"\b{re.escape(keyword)}\b"

            if re.search(pattern, text):
                topics.append(topic)
                break

    return topics

In [247]:
test_comments = [
    "the product arrived damaged and the seller was very helpful",
    "my order did not arrive and I am still waiting",
    "great product and excellent quality",
    "the package was well packed",
    "very good service from the seller"
]

for text in test_comments:
    print(f"\nComment: {text}")
    print(f"Topics: {classify_topics(text, topic_keywords)}")


Comment: the product arrived damaged and the seller was very helpful
Topics: ['Delivery & Logistics', 'Product & Quality', 'Seller & Service']

Comment: my order did not arrive and I am still waiting
Topics: ['Delivery & Logistics', 'Order & Purchase']

Comment: great product and excellent quality
Topics: ['Product & Quality']

Comment: the package was well packed
Topics: []

Comment: very good service from the seller
Topics: ['Seller & Service']


In [300]:
comments["topics"] = comments["clean_text"].apply(
    lambda text: classify_topics(text, topic_keywords)
)

comments["topic_count"] = comments["topics"].str.len()

In [302]:
print(f"Comments: {len(comments):,}")
print(f"Unique review records: {comments['id_order_reviews'].nunique():,}")
print(f"Unique review IDs: {comments['review_id'].nunique():,}")

Comments: 40,886
Unique review records: 40,886
Unique review IDs: 40,577


In [301]:
comments[
    [
        "review_score",
        "review_comment_message_en",
        "topics"
    ]
].head(20)

,review_score,review_comment_message_en,topics
3,5,i received it well before the promised deadline.,[Delivery & Logistics]
4,5,"congratulations, lannister stores! i loved buy...",[]
9,4,"efficient device. on the site, the devices bra...",[Delivery & Logistics]
12,4,"but a little bit, its lagging... for the price...",[]
15,5,"reliable seller, product ok, and delivery befo...","[Delivery & Logistics, Seller & Service]"
16,2,"i would like to know what happened, i always r...","[Delivery & Logistics, Order & Purchase]"
19,1,terrible,[]
22,5,store,[Seller & Service]
24,5,thank you for the attention you gave me.,[Seller & Service]
27,5,the purchase was made easily. the delivery was...,"[Delivery & Logistics, Order & Purchase]"


In [250]:
comments["topic_count"] = comments["topics"].str.len()

comments["topic_count"].value_counts().sort_index()

topic_count
0    10556
1    17967
2    10131
3     2064
4      163
5        5
Name: count, dtype: int64

In [251]:
topic_counts = Counter(
    topic
    for topics in comments["topics"]
    for topic in topics
)

topic_counts.most_common()

[('Delivery & Logistics', 23446),
 ('Order & Purchase', 7868),
 ('Product & Quality', 7725),
 ('Seller & Service', 4764),
 ('Packaging', 1295)]

In [306]:
unclassified = comments[
    comments["topic_count"] == 0
].copy()

print(f"Unclassified comments: {len(unclassified):,}")
print(f"Share: {len(unclassified) / len(comments):.2%}")

Unclassified comments: 10,556
Share: 25.82%


In [253]:
unclassified[
    [
        "review_score",
        "review_comment_message_en"
    ]
].sample(30, random_state=42)

,review_score,review_comment_message_en
17886,5,i recommend it.
76328,5,real product.
78015,5,great.
12496,4,"the sole is too high for a yearold child, it c..."
60761,4,"one of the rods was missing the line fitting, ..."
28347,4,good
89579,1,"very weak fragrance, the product appears to be..."
72289,5,excellent.
84388,4,everything is fine.
50861,5,excellent


In [254]:
unclassified_words = Counter(
    word
    for text in unclassified["clean_text"]
    for word in text.split()
    if word not in stop_words
)

unclassified_words.most_common(30)

[('good', 2138),
 ('very', 2051),
 ('product', 1828),
 ('recommend', 1225),
 ('great', 890),
 ('everything', 677),
 ('not', 642),
 ('loved', 627),
 ('liked', 624),
 ('beautiful', 583),
 ('excellent', 455),
 ('came', 401),
 ('satisfied', 337),
 ('like', 288),
 ('ok', 276),
 ('one', 270),
 ('all', 261),
 ('perfect', 251),
 ('thank', 239),
 ('expectations', 238),
 ('expected', 234),
 ('just', 227),
 ('only', 220),
 ('highly', 217),
 ('congratulations', 213),
 ('no', 208),
 ('am', 197),
 ('lot', 195),
 ('didnt', 193),
 ('well', 187)]

In [255]:
unclassified_bigrams = Counter()

for text in unclassified["clean_text"]:
    words = [
        word
        for word in text.split()
        if word not in stop_words
    ]
    
    bigrams = zip(words, words[1:])
    
    unclassified_bigrams.update(
        " ".join(bigram)
        for bigram in bigrams
    )

unclassified_bigrams.most_common(30)

[('very good', 1146),
 ('very beautiful', 164),
 ('highly recommend', 164),
 ('liked product', 142),
 ('liked lot', 139),
 ('very satisfied', 137),
 ('everything fine', 134),
 ('all good', 114),
 ('met expectations', 111),
 ('really liked', 111),
 ('loved product', 111),
 ('does not', 92),
 ('good recommend', 90),
 ('didnt like', 85),
 ('recommend everyone', 82),
 ('everything perfect', 76),
 ('product not', 76),
 ('did not', 74),
 ('everything ok', 73),
 ('product came', 71),
 ('product very', 66),
 ('product good', 64),
 ('everything went', 61),
 ('very much', 60),
 ('am satisfied', 59),
 ('value money', 58),
 ('do not', 56),
 ('highly recommended', 53),
 ('very well', 52),
 ('am very', 50)]

In [256]:
validation_delivery = comments[
    comments["topics"].apply(
        lambda topics: "Delivery & Logistics" in topics
    )
][
    ["review_score", "review_comment_message_en", "topics"]
].sample(
    20,
    random_state=42
)

validation_delivery

,review_score,review_comment_message_en,topics
23931,5,"everything was normal regarding the sale, purc...","[Delivery & Logistics, Order & Purchase]"
1002,1,the product was not delivered!,[Delivery & Logistics]
22286,2,the product arrived with the side paint peelin...,[Delivery & Logistics]
55067,1,"i did not receive the products, nor the refund...","[Delivery & Logistics, Seller & Service, Order..."
53891,5,"super fast delivery, well before the deadline....",[Delivery & Logistics]
32744,5,delivery made earlier than expected. i liked i...,[Delivery & Logistics]
91864,5,"the product is good, i received it on time.",[Delivery & Logistics]
16612,5,very good! delivery occurred before expected.,[Delivery & Logistics]
34899,4,"the suitcase arrived perfectly, even before th...",[Delivery & Logistics]
27100,1,i will not buy from this store again. the deli...,"[Delivery & Logistics, Seller & Service, Order..."


In [257]:
validation_product = comments[
    comments["topics"].apply(
        lambda topics: "Product & Quality" in topics
    )
][
    ["review_score", "review_comment_message_en", "topics"]
].sample(
    20,
    random_state=42
)

validation_product

,review_score,review_comment_message_en,topics
79957,5,ontime delivery and excellent material quality,"[Delivery & Logistics, Product & Quality]"
29289,5,super fast delivery and great product.,"[Delivery & Logistics, Product & Quality]"
90061,5,speed and efficiency in delivery. product of g...,"[Delivery & Logistics, Product & Quality]"
13588,5,great product,[Product & Quality]
83157,5,fast delivery. quality product,"[Delivery & Logistics, Product & Quality]"
86819,4,"super competent company, delivered well before...","[Delivery & Logistics, Product & Quality]"
12865,5,"the product is very good, it was delivered bef...","[Delivery & Logistics, Product & Quality, Sell..."
34669,4,it arrived on time and the product is very good.,"[Delivery & Logistics, Product & Quality]"
17245,3,the mosquito net fabric is good quality and be...,"[Product & Quality, Order & Purchase]"
30121,5,"product delivered before the deadline, in exce...","[Delivery & Logistics, Product & Quality]"


In [258]:
def explain_topics(text, topic_keywords):
    results = {}

    for topic, keywords in topic_keywords.items():
        matched_keywords = []

        for keyword in keywords:
            pattern = rf"\b{re.escape(keyword)}\b"

            if re.search(pattern, text):
                matched_keywords.append(keyword)

        if matched_keywords:
            results[topic] = matched_keywords

    return results

In [259]:
test_text = "everything is fine, i do recommend it"

explain_topics(test_text, topic_keywords)

{}

In [260]:
test_text = "everything went as promised and expected"

explain_topics(test_text, topic_keywords)

{}

In [261]:
test_text = "the delivery is within the deadline"

explain_topics(test_text, topic_keywords)

{'Delivery & Logistics': ['delivery', 'deadline']}

In [262]:
validation_seller = comments[
    comments["topics"].apply(
        lambda topics: "Seller & Service" in topics
    )
][
    ["review_score", "review_comment_message_en", "topics"]
].sample(
    20,
    random_state=42
)

validation_seller

,review_score,review_comment_message_en,topics
16719,1,i do not recommend the store. they are adverti...,[Seller & Service]
21565,5,"excellent store, i have always bought from the...","[Seller & Service, Order & Purchase]"
24893,4,great customer service,[Seller & Service]
53912,5,i loved the store,[Seller & Service]
29129,5,"great service, the delivery deadline exceeded ...","[Delivery & Logistics, Product & Quality, Sell..."
90439,5,very good seller. i recommend. .,[Seller & Service]
9904,3,the washers i bought along with it were missin...,"[Delivery & Logistics, Seller & Service, Order..."
12748,3,"my product, which was a computer power supply,...","[Delivery & Logistics, Seller & Service]"
27853,1,i would like to know what happened. i have bee...,[Seller & Service]
54047,5,best store in brazil.,[Seller & Service]


In [263]:
validation_order = comments[
    comments["topics"].apply(
        lambda topics: "Order & Purchase" in topics
    )
][
    ["review_score", "review_comment_message_en", "topics"]
].sample(
    20,
    random_state=42
)

validation_order

,review_score,review_comment_message_en,topics
63133,5,"best purchase ive made, product better than ad...",[Order & Purchase]
51600,5,it is incredible that for the first time a del...,"[Delivery & Logistics, Order & Purchase]"
19356,5,"product of excellent quality, the order arrive...","[Delivery & Logistics, Product & Quality, Orde..."
86575,1,i bought blankets and the store only sent me o...,"[Seller & Service, Order & Purchase]"
19667,1,i bought minecraft cm shiny blue swords. i onl...,"[Delivery & Logistics, Product & Quality, Orde..."
48909,2,terrible and horrible taste! impossible to dri...,[Order & Purchase]
4185,4,the only issue was a notice i received saying ...,"[Delivery & Logistics, Order & Purchase]"
53083,1,"half of the order was missing, that is, the ei...",[Order & Purchase]
47002,1,a day delivery delay is when you request infor...,"[Delivery & Logistics, Order & Purchase]"
42531,1,i bought two glasses and two earrings came i w...,[Order & Purchase]


In [264]:
validation_packaging = comments[
    comments["topics"].apply(
        lambda topics: "Packaging" in topics
    )
][
    ["review_score", "review_comment_message_en", "topics"]
].sample(
    20,
    random_state=42
)

validation_packaging

,review_score,review_comment_message_en,topics
65534,3,"i havent used it, its packaged. when i need it...",[Packaging]
25841,5,very good... packaging was perfect... quality ...,"[Product & Quality, Packaging]"
66176,1,"being a ceramic product, it is fragile. when u...","[Delivery & Logistics, Packaging]"
86201,5,its very beautiful! it arrived well packaged a...,"[Delivery & Logistics, Packaging]"
75180,3,"there was only one problem with the packaging,...","[Delivery & Logistics, Packaging]"
42145,1,"the product is of very poor quality, it was se...","[Delivery & Logistics, Product & Quality, Pack..."
37801,5,"great product as expected, very well packaged,...","[Delivery & Logistics, Product & Quality, Pack..."
82640,1,"product delivered poorly packaged, without a f...","[Delivery & Logistics, Product & Quality, Sell..."
69235,5,well packaged and delivered on time,"[Delivery & Logistics, Packaging]"
5565,5,delivery was well before expected. the product...,"[Delivery & Logistics, Product & Quality, Pack..."


In [265]:
packaging_test_comments = [
    "the product arrived adequately packaged",
    "the product was very well packaged",
    "great packaging",
    "the packaging was damaged",
    "i ordered boxes and only received one",
    "i bought two boxes and only received one"
]

for text in packaging_test_comments:
    cleaned = clean_text(text)

    print(f"\nComment: {text}")
    print(f"Topics: {classify_topics(cleaned, topic_keywords)}")


Comment: the product arrived adequately packaged
Topics: ['Delivery & Logistics', 'Packaging']

Comment: the product was very well packaged
Topics: ['Packaging']

Comment: great packaging
Topics: ['Packaging']

Comment: the packaging was damaged
Topics: ['Packaging']

Comment: i ordered boxes and only received one
Topics: ['Delivery & Logistics']

Comment: i bought two boxes and only received one
Topics: ['Delivery & Logistics', 'Order & Purchase']


In [266]:
product_test_comments = [
    "its good, but its not aluminum",
    "i thought it would be a bit thicker",
    "the product is very good",
    "the umbrella came with small holes",
    "the hose burst before even being used",
    "this product is much inferior",
    "good product",
    "excellent product",
    "i received the product before the deadline"
]

for text in product_test_comments:
    cleaned = clean_text(text)

    print(f"\nComment: {text}")
    print(f"Topics: {classify_topics(cleaned, topic_keywords)}")


Comment: its good, but its not aluminum
Topics: ['Product & Quality']

Comment: i thought it would be a bit thicker
Topics: ['Product & Quality']

Comment: the product is very good
Topics: ['Product & Quality']

Comment: the umbrella came with small holes
Topics: ['Product & Quality']

Comment: the hose burst before even being used
Topics: ['Product & Quality']

Comment: this product is much inferior
Topics: ['Product & Quality']

Comment: good product
Topics: ['Product & Quality']

Comment: excellent product
Topics: ['Product & Quality']

Comment: i received the product before the deadline
Topics: ['Delivery & Logistics']


In [307]:
topic_rows = []

for _, row in comments.iterrows():

    for topic in row["topics"]:

        topic_rows.append({
            "id_order_reviews": row["id_order_reviews"],
            "review_id": row["review_id"],
            "order_id": row["order_id"],
            "review_score": row["review_score"],
            "review_comment_message_en": row["review_comment_message_en"],
            "topic": topic
        })

comment_topics = pd.DataFrame(topic_rows)

In [308]:
comment_topics.shape

(45098, 6)

In [309]:
comment_topics.head(20)

,id_order_reviews,review_id,order_id,review_score,review_comment_message_en,topic
0,4,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,i received it well before the promised deadline.,Delivery & Logistics
1,10,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,"efficient device. on the site, the devices bra...",Delivery & Logistics
2,16,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,"reliable seller, product ok, and delivery befo...",Delivery & Logistics
3,16,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,"reliable seller, product ok, and delivery befo...",Seller & Service
4,17,9314d6f9799f5bfba510cc7bcd468c01,0dacf04c5ad59fd5a0cc1faa07c34e39,2,"i would like to know what happened, i always r...",Delivery & Logistics
5,17,9314d6f9799f5bfba510cc7bcd468c01,0dacf04c5ad59fd5a0cc1faa07c34e39,2,"i would like to know what happened, i always r...",Order & Purchase
6,23,d21bbc789670eab777d27372ab9094cc,4fc44d78867142c627497b60a7e0228a,5,store,Seller & Service
7,25,0e0190b9db53b689b285d3f3916f8441,79832b7cb59ac6f887088ffd686e1d5e,5,thank you for the attention you gave me.,Seller & Service
8,28,fe3db7c069d694bab50cc43463f91608,2ca73e2ff9e3a186ad1e1ffb9b1d9c10,5,the purchase was made easily. the delivery was...,Delivery & Logistics
9,28,fe3db7c069d694bab50cc43463f91608,2ca73e2ff9e3a186ad1e1ffb9b1d9c10,5,the purchase was made easily. the delivery was...,Order & Purchase


In [310]:
print(
    "Duplicate review-topic pairs:",
    comment_topics.duplicated(
        ["id_order_reviews", "topic"]
    ).sum()
)

Duplicate review-topic pairs: 0


In [311]:
topic_summary = (
    comment_topics
    .groupby("topic")
    .agg(
        review_count=("review_id", "nunique"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

topic_summary["share_of_comments"] = (
    topic_summary["review_count"] / len(comments)
)

topic_summary = topic_summary.sort_values(
    "review_count",
    ascending=False
)

topic_summary

,topic,review_count,avg_review_score,share_of_comments
0,Delivery & Logistics,23258,3.428303,0.568850
1,Order & Purchase,7799,2.746568,0.190750
3,Product & Quality,7669,4.052427,0.187570
4,Seller & Service,4731,3.383501,0.115712
2,Packaging,1288,4.101158,0.031502


In [312]:
comment_topics["is_negative"] = (
    comment_topics["review_score"] <= 2
)

In [313]:
topic_performance = (
    comment_topics
    .groupby("topic")
    .agg(
        review_count=("id_order_reviews", "nunique"),
        negative_reviews=("is_negative", "sum"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

topic_performance["negative_rate"] = (
    topic_performance["negative_reviews"]
    / topic_performance["review_count"]
)

topic_performance.sort_values(
    "negative_rate",
    ascending=False
)

,topic,review_count,negative_reviews,avg_review_score,negative_rate
1,Order & Purchase,7868,4030,2.746568,0.512201
4,Seller & Service,4764,1693,3.383501,0.355374
0,Delivery & Logistics,23446,7841,3.428303,0.334428
3,Product & Quality,7725,1291,4.052427,0.167120
2,Packaging,1295,198,4.101158,0.152896


In [314]:
topic_analysis = (
    topic_performance
    .merge(
        topic_summary[
            ["topic", "share_of_comments"]
        ],
        on="topic",
        how="left"
    )
)

topic_analysis = topic_analysis[
    [
        "topic",
        "review_count",
        "share_of_comments",
        "negative_reviews",
        "negative_rate",
        "avg_review_score"
    ]
].sort_values(
    "review_count",
    ascending=False
)

topic_analysis

,topic,review_count,share_of_comments,negative_reviews,negative_rate,avg_review_score
0,Delivery & Logistics,23446,0.568850,7841,0.334428,3.428303
1,Order & Purchase,7868,0.190750,4030,0.512201,2.746568
3,Product & Quality,7725,0.187570,1291,0.167120,4.052427
4,Seller & Service,4764,0.115712,1693,0.355374,3.383501
2,Packaging,1295,0.031502,198,0.152896,4.101158


In [292]:
reviews["id_order_reviews"].nunique()

99224

In [293]:
len(reviews)

99224

In [294]:
reviews["review_id"].nunique()

98410

In [315]:
topic_performance["share_of_comments"] = (
    topic_performance["review_count"]
    / len(comments)
)

In [316]:
topic_performance

,topic,review_count,negative_reviews,avg_review_score,negative_rate,share_of_comments
0,Delivery & Logistics,23446,7841,3.428303,0.334428,0.573448
1,Order & Purchase,7868,4030,2.746568,0.512201,0.192438
2,Packaging,1295,198,4.101158,0.152896,0.031673
3,Product & Quality,7725,1291,4.052427,0.167120,0.188940
4,Seller & Service,4764,1693,3.383501,0.355374,0.116519


In [317]:
topic_performance["share_pct"] = (
    topic_performance["share_of_comments"] * 100
)

topic_performance["negative_rate_pct"] = (
    topic_performance["negative_rate"] * 100
)

topic_performance

,topic,review_count,negative_reviews,avg_review_score,negative_rate,share_of_comments,share_pct,negative_rate_pct
0,Delivery & Logistics,23446,7841,3.428303,0.334428,0.573448,57.344812,33.442805
1,Order & Purchase,7868,4030,2.746568,0.512201,0.192438,19.243751,51.220132
2,Packaging,1295,198,4.101158,0.152896,0.031673,3.167343,15.289575
3,Product & Quality,7725,1291,4.052427,0.167120,0.188940,18.893998,16.711974
4,Seller & Service,4764,1693,3.383501,0.355374,0.116519,11.651910,35.537364


In [318]:
topic_score_distribution = (
    comment_topics
    .groupby(["topic", "review_score"])
    .agg(
        review_count=("id_order_reviews", "nunique")
    )
    .reset_index()
)

topic_score_distribution

,topic,review_score,review_count
0,Delivery & Logistics,1,6424
1,Delivery & Logistics,2,1417
2,Delivery & Logistics,3,1966
3,Delivery & Logistics,4,2971
4,Delivery & Logistics,5,10668
5,Order & Purchase,1,3340
6,Order & Purchase,2,690
7,Order & Purchase,3,785
8,Order & Purchase,4,730
9,Order & Purchase,5,2323


In [319]:
topic_score_distribution["score_share"] = (
    topic_score_distribution["review_count"]
    /
    topic_score_distribution
    .groupby("topic")["review_count"]
    .transform("sum")
)

topic_score_distribution

,topic,review_score,review_count,score_share
0,Delivery & Logistics,1,6424,0.273991
1,Delivery & Logistics,2,1417,0.060437
2,Delivery & Logistics,3,1966,0.083852
3,Delivery & Logistics,4,2971,0.126717
4,Delivery & Logistics,5,10668,0.455003
5,Order & Purchase,1,3340,0.424504
6,Order & Purchase,2,690,0.087697
7,Order & Purchase,3,785,0.099771
8,Order & Purchase,4,730,0.092781
9,Order & Purchase,5,2323,0.295247


In [320]:
overall_negative_reviews = (
    comments["review_score"] <= 2
).sum()

overall_negative_rate = (
    overall_negative_reviews / len(comments)
)

print(f"Negative reviews: {overall_negative_reviews:,}")
print(f"Overall negative rate: {overall_negative_rate:.2%}")

Negative reviews: 10,883
Overall negative rate: 26.62%


In [322]:
topic_performance["negative_share"] = (
    topic_performance["negative_reviews"]
    / topic_performance["negative_reviews"].sum()
)

topic_performance[
    [
        "topic",
        "review_count",
        "negative_reviews",
        "negative_rate",
        "negative_share"
    ]
].sort_values(
    "negative_reviews",
    ascending=False
)

,topic,review_count,negative_reviews,negative_rate,negative_share
0,Delivery & Logistics,23446,7841,0.334428,0.520893
1,Order & Purchase,7868,4030,0.512201,0.267721
4,Seller & Service,4764,1693,0.355374,0.112469
3,Product & Quality,7725,1291,0.167120,0.085764
2,Packaging,1295,198,0.152896,0.013154


In [323]:
classified_comments = (
    comments["topic_count"] > 0
).sum()

unclassified_comments = (
    comments["topic_count"] == 0
).sum()

classification_coverage = (
    classified_comments / len(comments)
)

print(f"Classified comments: {classified_comments:,}")
print(f"Unclassified comments: {unclassified_comments:,}")
print(f"Classification coverage: {classification_coverage:.2%}")

Classified comments: 30,330
Unclassified comments: 10,556
Classification coverage: 74.18%


In [325]:
coverage_by_score = (
    comments
    .groupby("review_score")
    .agg(
        total_reviews=("id_order_reviews", "nunique"),
        classified_reviews=(
            "topic_count",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

coverage_by_score["coverage"] = (
    coverage_by_score["classified_reviews"]
    / coverage_by_score["total_reviews"]
)

coverage_by_score

,review_score,total_reviews,classified_reviews,coverage
0,1,8740,7718,0.883066
1,2,2143,1807,0.843210
2,3,3555,2618,0.736428
3,4,5956,4038,0.677972
4,5,20492,14149,0.690465


In [326]:
final_topic_analysis = (
    topic_performance[
        [
            "topic",
            "review_count",
            "share_pct",
            "negative_reviews",
            "negative_rate_pct",
            "avg_review_score"
        ]
    ]
    .sort_values(
        "negative_reviews",
        ascending=False
    )
    .reset_index(drop=True)
)

final_topic_analysis

,topic,review_count,share_pct,negative_reviews,negative_rate_pct,avg_review_score
0,Delivery & Logistics,23446,57.344812,7841,33.442805,3.428303
1,Order & Purchase,7868,19.243751,4030,51.220132,2.746568
2,Seller & Service,4764,11.651910,1693,35.537364,3.383501
3,Product & Quality,7725,18.893998,1291,16.711974,4.052427
4,Packaging,1295,3.167343,198,15.289575,4.101158


In [327]:
final_topic_analysis["negative_contribution_pct"] = (
    final_topic_analysis["negative_reviews"]
    / final_topic_analysis["negative_reviews"].sum()
    * 100
)

final_topic_analysis

,topic,review_count,share_pct,negative_reviews,negative_rate_pct,avg_review_score,negative_contribution_pct
0,Delivery & Logistics,23446,57.344812,7841,33.442805,3.428303,52.089285
1,Order & Purchase,7868,19.243751,4030,51.220132,2.746568,26.772072
2,Seller & Service,4764,11.651910,1693,35.537364,3.383501,11.246928
3,Product & Quality,7725,18.893998,1291,16.711974,4.052427,8.576364
4,Packaging,1295,3.167343,198,15.289575,4.101158,1.315352


In [329]:
final_topic_analysis["priority_score"] = (
    final_topic_analysis["negative_rate_pct"]
    * final_topic_analysis["review_count"]
)

final_topic_analysis.sort_values(
    "priority_score",
    ascending=False
)

,topic,review_count,share_pct,negative_reviews,negative_rate_pct,avg_review_score,negative_contribution_pct,priority_score
0,Delivery & Logistics,23446,57.344812,7841,33.442805,3.428303,52.089285,784100.0
1,Order & Purchase,7868,19.243751,4030,51.220132,2.746568,26.772072,403000.0
2,Seller & Service,4764,11.651910,1693,35.537364,3.383501,11.246928,169300.0
3,Product & Quality,7725,18.893998,1291,16.711974,4.052427,8.576364,129100.0
4,Packaging,1295,3.167343,198,15.289575,4.101158,1.315352,19800.0


In [330]:
topic_score_share = (
    comment_topics
    .groupby(["topic", "review_score"])
    .agg(
        review_count=("id_order_reviews", "nunique")
    )
    .reset_index()
)

topic_score_share["score_share"] = (
    topic_score_share["review_count"]
    /
    topic_score_share
    .groupby("topic")["review_count"]
    .transform("sum")
)

topic_score_share

,topic,review_score,review_count,score_share
0,Delivery & Logistics,1,6424,0.273991
1,Delivery & Logistics,2,1417,0.060437
2,Delivery & Logistics,3,1966,0.083852
3,Delivery & Logistics,4,2971,0.126717
4,Delivery & Logistics,5,10668,0.455003
5,Order & Purchase,1,3340,0.424504
6,Order & Purchase,2,690,0.087697
7,Order & Purchase,3,785,0.099771
8,Order & Purchase,4,730,0.092781
9,Order & Purchase,5,2323,0.295247


In [332]:
comment_topics.to_csv(
    "../datasets/processed/comment_topics.csv",
    index=False
)

In [336]:
final_topic_analysis.to_csv(
    "../datasets/processed/final_topic_analysis.csv",
    index=False
)

In [337]:
topic_score_distribution.to_csv(
    "../datasets/processed/topic_score_distribution.csv",
    index=False
)